<a href="https://colab.research.google.com/github/Dgowda11/llama-spam-finetuning/blob/main/notebooks/05_base_model_baseline_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this phase we will assess how well the original, unfine-tuned Llama 3.2 3B Instruct model performs before QLoRA.

In [1]:

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())

    total_memory = torch.cuda.get_device_properties(0).total_memory
    free_memory, total_memory_cuda = torch.cuda.mem_get_info()

    print(f"Total GPU Memory: {total_memory / 1024**3:.2f} GB")
    print(f"Free GPU Memory: {free_memory / 1024**3:.2f} GB")
else:
    print("GPU is not available. Check your Colab runtime settings.")


PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Count: 1
Total GPU Memory: 14.56 GB
Free GPU Memory: 14.46 GB


In [2]:
# install required Dependencies

%pip install -q -U transformers accelerate datasets huggingface_hub kaggle scikit-learn pandas matplotlib seaborn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
%pip install -q --force-reinstall pandas==2.2.3 pyarrow==23.0.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.2.0 requires pyarrow>=24.0.0, but you have pyarrow 23.0.1 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.


In [4]:

# Clone the project repository

%cd /content

!git clone https://github.com/Dgowda11/sms-spam-llama-qlora.git

%cd /content/sms-spam-llama-qlora

!ls


/content
Cloning into 'sms-spam-llama-qlora'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 88 (delta 37), reused 56 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 612.89 KiB | 13.32 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/sms-spam-llama-qlora
data  LICENSE  notebooks  README.md  requirements.txt  src


In [5]:

import torch
import transformers
import datasets
import pandas as pd
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("CUDA Available:", torch.cuda.is_available())


PyTorch: 2.14.1+cu130
Transformers: 5.23.0
Datasets: 5.2.0
Pandas: 2.2.3
Scikit-learn: 1.9.1
CUDA Available: True


In [6]:
#Authenticate the Hugging Face
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful!")

Hugging Face authentication successful!


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)
print("Tokenizer loaded successfully!")
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Tokenizer: TokenizersBackend
Vocabulary size: 128000
EOS Token: <|eot_id|>


Load the Base Model

In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Base Llama model loaded successfully!")
print("Model dtype:", model.dtype)
print("Model device:", model.device)
print("Model training mode:", model.training)
print("Device mapping:", model.hf_device_map)

In [10]:
# MOnitor the Resources
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    free, total = torch.cuda.mem_get_info(0)

    print(f"GPU Allocated: {allocated:.2f} GB")
    print(f"GPU Reserved: {reserved:.2f} GB")
    print(f"GPU Free: {free / 1024**3:.2f} GB")
    print(f"GPU Total: {total / 1024**3:.2f} GB")

GPU Allocated: 5.98 GB
GPU Reserved: 6.00 GB
GPU Free: 8.46 GB
GPU Total: 14.56 GB
